In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

In [5]:
df = pd.read_csv('data/datasets_TFM + diccionario/customer_sociodemographics.csv', index_col=0)

Para inferir 25 géneros: solamente 25 nulos que corresponden a dos clientes de la región 28.0. 

Cogemos la media de salario para esa edad e inferimos el género en función de lo que se aproxime a la media.

In [6]:
df[df['pk_cid']==476023] = df[df['pk_cid']==476023].fillna('H')
df[df['pk_cid']==216507] = df[df['pk_cid']==216507].fillna('V')

Para inferir el 2264 region_codes: los paises que no son 'ES' no tienen region_code (inferimos con -1), y vemos cómo inferimos los de 'ES'...


In [7]:
df['region_code'] = df['region_code'].fillna(-1)

Solamente 1 cliente de género 'V' y 45 años que entró en 2018-01: casi el 30% de los 'V' de 45 años que entraron el 2018-01 son de la región 28.0

In [8]:
df.loc[df['pk_cid']==1234433,'region_code'] = 28.0

Nulos en 'salary': 1.541.104 ('ES' 1.538.889/ 5.960.672)(No 'ES' 2.215/ 2.252)

Les damos valor desconocido, por si posteriormente podemos inferirlo con otras variables

In [9]:
df['salary'] = df['salary'].fillna('unknown')

In [10]:
# convertimos variable id en string, y resto de variables no numéricas a categóricas
for col in ['country_id', 'region_code', 'gender', 'deceased']:
    df[col] = df[col].astype('category')
df['pk_cid'] = df['pk_cid'].astype('str')

In [11]:
# Comprobamos para qué clientes hay variaciones:
df_compr = df.copy()

# Lista de campos que a comprobar
campos = ['country_id', 'region_code', 'gender', 'salary']

# Agrupar por cliente y contar valores únicos por campo
consistencia = df_compr.groupby('pk_cid')[campos].nunique() == 1

# Añadir columna que indica si todos los campos son constantes
consistencia['todos_constantes'] = consistencia.all(axis=1)

# Filtrar clientes con alguna variación en los campos
clientes_con_variaciones = consistencia[~consistencia['todos_constantes']]

In [12]:
clientes_con_variaciones = clientes_con_variaciones.reset_index()

In [13]:
df_final = pd.merge(df, clientes_con_variaciones, on='pk_cid', how='left')
df_final.drop(['country_id_y', 'region_code_y', 'gender_y', 'salary_y'], axis=1, inplace=True)

In [14]:
df_final.rename(columns={"country_id_x": "country_id", "region_code_x": "region_code", "gender_x": "gender", "salary_x": "salary", "todos_constantes": "z_constant"}, inplace=True)

In [15]:
df_final['z_constant'] = df_final['z_constant'].fillna(True)

/var/folders/36/m07hd80d0cq09shsjpwrlm9m0000gn/T/ipykernel_46467/2681614693.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final['z_constant'] = df_final['z_constant'].fillna(True)


In [16]:
df_final.to_csv('data/pbi_datasets/customer_sociodemographics_pbi.csv', sep=',')